
Segmentation Lead: Diabetes Risk Clustering

K-Means clustering with k=3 to identify lifestyle personas


Import Libraries

In [ ]:
import pandas as pd                    
import seaborn as sns                  
import matplotlib.pyplot as plt        
from sklearn.preprocessing import StandardScaler   
from sklearn.cluster import KMeans                
from sklearn.decomposition import PCA              
import os

Load Dataset (with robust path handling)

In [ ]:
# Get the directory where this script is located
script_dir = os.path.dirname(os.path.abspath(__file__))
# Go up one level to project root, then into data folder
data_path = os.path.join(script_dir, '..', 'data', 'Diabetes_and_LifeStyle_Dataset_.csv')
df = pd.read_csv(data_path)

print("First 5 rows of the dataset:")
print(df.head())
print("\nDataset shape:", df.shape)

Select Lifestyle Features for Clustering

In [ ]:
features = [
    'physical_activity_minutes_per_week',   
    'diet_score',                           
    'sleep_hours_per_day',                  
    'bmi',                                  
    'alcohol_consumption_per_week'          
]

print("\nLifestyle features (first 5 rows):")
print(df[features].head())

In [ ]:
#Scale the Features (Important for K-Means)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

print("\nAfter scaling (first 5 rows):")
print(X_scaled[:5])

In [ ]:
#Run K-Means Clustering with k=3

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Lifestyle_Segment'] = kmeans.fit_predict(X_scaled)

print("\nNumber of people in each cluster:")
print(df['Lifestyle_Segment'].value_counts())

In [ ]:
#Assign Meaningful Persona Names


persona_map = {
    0: 'The Active Movers',
    1: 'The High-Risk Inactive',
    2: 'The Mindful Eaters'
}
df['Persona'] = df['Lifestyle_Segment'].map(persona_map)

print("\nFirst 10 rows with Persona assigned:")
print(df[['Lifestyle_Segment', 'Persona'] + features].head(10))

In [ ]:
# PCA for 2D Visualisation


pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df['PC1'] = pca_results[:, 0]
df['PC2'] = pca_results[:, 1]

print(f"\nPC1 explains: {pca.explained_variance_ratio_[0] * 100:.2f}%")
print(f"PC2 explains: {pca.explained_variance_ratio_[1] * 100:.2f}%")
print(f"Total variance explained: {(pca.explained_variance_ratio_.sum() * 100):.2f}%")

In [ ]:
#Visualisation: Personas vs Clinical Reality

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

persona_colors = {
    'The Active Movers': 'green',
    'The High-Risk Inactive': 'red',
    'The Mindful Eaters': 'blue'
}

disease_colors = {
    'No Diabetes': 'green',
    'Pre-Diabetes': 'orange',
    'Type 2': 'red'
}

# Graph A: Lifestyle Personas
sns.scatterplot(data=df, x='PC1', y='PC2', hue='Persona', 
                palette=persona_colors, alpha=0.4, s=30, ax=axes[0])
axes[0].set_title("1. LIFESTYLE PERSONAS\n(Groups defined by Habits)", fontsize=15, fontweight='bold')
axes[0].legend(title="Persona")

# Graph B: Reality
target_stages = ['No Diabetes', 'Pre-Diabetes', 'Type 2']
df_filtered = df[df['diabetes_stage'].isin(target_stages)]
sns.scatterplot(data=df_filtered, x='PC1', y='PC2', hue='diabetes_stage', 
                palette=disease_colors, alpha=0.4, s=30, ax=axes[1])
axes[1].set_title("2. CLINICAL REALITY\n(Actual Diabetes Diagnosis)", fontsize=15, fontweight='bold')
axes[1].legend(title="Diabetes Stage")

plt.tight_layout()
plt.show()

In [ ]:
#Risk Analysis: Two Views


# Row percentages (within each persona)
row_table = pd.crosstab(df['Persona'], df['diabetes_stage'], normalize='index') * 100
print("\n" + "="*60)
print("VIEW 1: WITHIN EACH PERSONA – what % has each diabetes stage?")
print("(Each row sums to 100%)")
print("="*60)
print(row_table[['No Diabetes', 'Pre-Diabetes', 'Type 2']].round(2))

# Column percentages (within each diabetes stage)
col_table = pd.crosstab(df['Persona'], df['diabetes_stage'], normalize='columns') * 100
print("\n" + "="*60)
print("VIEW 2: TOTAL POPULATION SPREAD – for each diabetes stage, what % are in each Persona?")
print("(Each column sums to 100%)")
print("="*60)
print(col_table[['No Diabetes', 'Pre-Diabetes', 'Type 2']].round(2))

print("\n" + "🔴"*30)
print("KEY INSIGHT: Distribution of Type 2 patients across personas")
print("🔴"*30)
type2_distribution = col_table['Type 2'].sort_values(ascending=False)
for persona, pct in type2_distribution.items():
    print(f"   {persona}: {pct:.1f}%")

In [ ]:
#Cluster Profiles (Average Values per Persona)

print("\n" + "="*60)
print("CLUSTER PROFILES (average lifestyle values per persona)")
print("="*60)
print(df.groupby('Persona')[features].mean().round(1))

In [ ]:
#Additional Bar Chart of Type 2 Capture Rates

plt.figure(figsize=(8, 5))
type2_capture = col_table['Type 2'].sort_values()
colors = ['#2ca02c', '#1f77b4', '#d62728']
plt.barh(type2_capture.index, type2_capture.values, color=colors)
plt.xlabel('Percentage of all Type 2 patients (%)')
plt.title('Which Persona Contains the Most Type 2 Diabetics?')
plt.tight_layout()
plt.show()

In [ ]:
#Save Results (Optional – commented out)


# df.to_csv('../artifacts/segmented_patients.csv', index=False)
# print("\n✓ Saved segmented dataset to '../artifacts/segmented_patients.csv'")
# cluster_profiles = df.groupby('Persona')[features].mean()
# cluster_profiles.to_csv('../artifacts/cluster_profiles.csv')
# print("✓ Saved cluster profiles to '../artifacts/cluster_profiles.csv'")

print("\n" + "="*60)
print("SEGMENTATION COMPLETE!! - Your personas are ready!")
print("="*60)